# ECG Signal Processing Workshop — Notebook 6: RPNet AI-Based R-Peak Detection

## Overview

Classical R-peak detectors rely on hand-crafted signal processing pipelines:
bandpass filters, derivative operators, adaptive thresholds. These work well
on clean clinical ECG but degrade rapidly on **noisy textile electrodes**
where baseline wander, motion artifacts, and low signal-to-noise ratios
distort the QRS morphology that classical rules depend on.

Deep learning offers a fundamentally different strategy: instead of
engineering features by hand, a convolutional neural network **learns**
discriminative features directly from annotated data. RPNet, introduced by
Peimankar & Puthusserypady (2021), is one such architecture — an
Inception-Residual U-Net that predicts a **distance transform map** from
raw ECG, turning R-peak detection into a dense regression problem.

### What this notebook covers

| Step | Description |
|---|---|
| **Architecture** | IncRes-UNet: Inception + Residual blocks inside a U-Net backbone |
| **Resampling** | Converting BIOPAC (1000 Hz) and Belt (100 Hz) signals to RPNet's native 500 Hz |
| **Inference** | Sliding-window prediction with overlap averaging |
| **Post-processing** | Distance-map → R-peak extraction via local minima |
| **Evaluation** | Comparison with classical detectors from NB04 |
| **Segment analysis** | Signal quality vs detection performance scatter plots |
| **Workshop summary** | Final detector hierarchy and HRV recommendations |

## 1. RPNet Architecture Deep Dive

### IncRes-UNet: Inception + Residual Blocks Inside a U-Net

RPNet's backbone is a **U-Net** — an encoder-decoder architecture with skip
connections that preserves fine-grained temporal information across the
bottleneck. What makes it distinctive is the use of **Inception-Residual
(IncRes) blocks** as the core building unit.

#### Inception blocks

Each IncRes block applies **four parallel convolutions** with different kernel
sizes (15, 17, 19, 21 samples), concatenates them, and adds a residual
connection:

```
Input (C channels)
  │
  ├──► Conv1d(k=15)   ──► BN ──► C/4 channels
  ├──► 1×1 → Conv1d(k=17) ──► BN ──► C/4 channels
  ├──► 1×1 → Conv1d(k=19) ──► BN ──► C/4 channels
  ├──► 1×1 → Conv1d(k=21) ──► BN ──► C/4 channels
  │
  └──► Concat (C channels) + Residual (1×1 conv) ──► ReLU ──► Output
```

The multi-scale kernels capture QRS complexes at different temporal
resolutions — narrow kernels detect sharp R-peaks while wider kernels
capture broader QRS morphology. This is critical for generalizing across
different heart rates and ECG morphologies.

#### Full architecture (ASCII diagram)

```
Input: [1, 1, 5000]  (1 channel, 10 seconds @ 500 Hz)
  │
  ▼
┌─── ENCODER ──────────────────────────────────────────────┐
│ e1: Conv(1→64,  k=4, s=2) + BN + IncRes(64)   → 2500   │
│ e2: Conv(64→128, k=4, s=2) + BN + IncRes(128)  → 1250   │
│ e2add: Conv(128→128, k=3, s=1) + BN            → 1250   │
│ e3: Conv(128→256, k=4, s=2) + BN + IncRes(256) →  625   │
│ e4: Conv(256→512, k=4, s=2) + BN + IncRes(512) →  312   │
│ e4add: Conv(512→512, k=3, s=1) + BN            →  312   │
│ e5: Conv(512→512, k=4, s=2) + BN + IncRes(512) →  156   │
│ e6: Conv(512→512, k=4, s=2) + BN + IncRes(512) →   78   │
│ e6add: Conv(512→512, k=3, s=1) + BN            →   78   │
│ e7: Conv(512→512, k=4, s=2) + BN + IncRes(512) →   39   │
│ e8: Conv(512→512, k=4, s=2) + BN               →   19   │
└──────────────────────────────────────────────────────────┘
  │
  ▼ Bottleneck: 512 × 19
  │
┌─── DECODER (with skip connections) ─────────────────────┐
│ d1:  ConvT(512→512, s=2) + IncRes  + skip(e7)  →   39  │
│ d2:  ConvT(1024→512, s=2) + IncRes + skip(e6a) →   78  │
│ d3:  ConvT(1024→512, s=1) + IncRes + skip(e6)  →   78  │
│ d4:  ConvT(1024→512, s=2) + IncRes + skip(e5)  →  156  │
│ d5:  ConvT(1024→512, s=2) + IncRes + skip(e4a) →  312  │
│ d6:  ConvT(1024→512, s=1) + IncRes + skip(e4)  →  312  │
│ d7:  ConvT(1024→256, s=2) + IncRes + skip(e3)  →  625  │
│ d8:  ConvT(512→128, s=2)           + skip(e2a) → 1250  │
│ d9:  ConvT(256→128, s=1)           + skip(e2)  → 1250  │
│ d10: ConvT(256→64,  s=2)           + skip(e1)  → 2500  │
│ out: ConvT(128→1,   s=2)                       → 5000  │
└─────────────────────────────────────────────────────────┘
  │
  ▼
Output: [1, 1, 5000]  (distance transform map)
```

#### Distance transform output

Unlike binary classifiers, RPNet outputs a **distance transform map** where
each sample's value represents its distance to the nearest R-peak. R-peak
locations correspond to **minima** (valleys) in this map — the value
approaches zero at peak locations and increases away from them.

#### Training details

- **Dataset**: MIT-BIH Arrhythmia Database (48 half-hour recordings, 360 Hz)
- **Preprocessing**: Upsampled from 360 Hz → 500 Hz
- **Input shape**: `[batch, 1, 5000]` — 10-second windows at 500 Hz
- **Loss**: MSE between predicted and true distance transform maps

## 2. Critical — Sampling Rate Mismatch

RPNet was trained on signals at **500 Hz**. Feeding signals at different
sampling rates will produce incorrect QRS morphology widths and break the
learned features.

| Signal | Native fs | Target | Resampling | `resample_poly` args |
|---|---|---|---|---|
| **BIOPAC** | 1000 Hz | 500 Hz | Downsample ×2 | `up=1, down=2` |
| **Belt** | 100 Hz | 500 Hz | Upsample ×5 | `up=5, down=1` |

After RPNet detects peaks in the 500 Hz domain, we convert peak indices
back to original sampling rates:

```
peak_original = round(peak_500hz × (fs_original / 500))
```

`resample_poly` from SciPy applies an anti-aliasing FIR filter before
downsampling and interpolation during upsampling, avoiding aliasing
artifacts that would confuse the network.

## 3. Setup & Configuration

### Model files

- **Model class**: `reference_codes/RPNet/network.py` (copied inline below)
- **Pre-trained weights**: Download from [Google Drive](https://drive.google.com/file/d/19xN7pZsALb09bxWjrSKdAlJmRqYL0M0g/view)
  and save to `reference_codes/RPNet/model.pt`

If the weights file is not found, the notebook will proceed with an
**untrained model** for demonstration purposes (predictions will be
meaningless, but the pipeline will run).

In [ ]:
# =====================================================================
# USER CONFIGURATION
# =====================================================================
BIOPAC_FILE_PATH = r"sample_data/BIOPAC_04020062_9_Female_1st.txt"
BELT_FILE_PATH   = r"sample_data/BABY_BELT_04020062_9_Female_1st.csv"
# --- Dataset format selector ---
DATASET_FORMAT = "BABY_BELT"  # Options: "BABY_BELT" or "CAREWEAR"
# --- CareWear paths ---
CAREWEAR_BIOPAC_FILE_PATH = r"sample_data/CAREWEAR/P10-stationary Bike1-biopac-01-30-2025.txt"
CAREWEAR_BELT_FILE_PATH   = r"sample_data/CAREWEAR/P10-stationary Bike1-belt-01-30-2025"
CAREWEAR_BIOPAC_ECG_COL   = "CH9"   # "CH9" (raw ECG) or "CH40" (AHA-filtered)
CAREWEAR_BELT_ECG_SCALE_FN = None   # e.g., lambda x: x * 0.001 for future scaling
OUTPUT_DIR       = r"outputs/NB06"

RPNET_MODEL_DIR  = r"reference_codes/RPNet"
RPNET_WEIGHTS    = r"reference_codes/RPNet/model.pt"
# =====================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import time
import os
import warnings

from scipy.signal import (
    butter, sosfiltfilt, iirnotch, tf2sos, medfilt,
    find_peaks, resample_poly,
)
from scipy.stats import kurtosis
from tqdm.notebook import tqdm

import neurokit2 as nk

try:
    import torch
    import torch.nn as nn
    TORCH_AVAILABLE = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"PyTorch {torch.__version__} — device: {DEVICE}")
except ImportError:
    TORCH_AVAILABLE = False
    DEVICE = "cpu"
    print("WARNING: PyTorch not installed. RPNet inference will be skipped.")
    print("Install with: pip install torch")

warnings.filterwarnings("ignore")
np.random.seed(42)

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory ready: {os.path.abspath(OUTPUT_DIR)}")

## 4. Data Loading & Preprocessing

Parsers and filter functions are copied inline (identical to NB01–NB04) so
this notebook is fully self-contained.

In [ ]:
# -----------------------------------------------------------------
# Parsers
# -----------------------------------------------------------------

def parse_biopac(filepath):
    """Parse BIOPAC AcqKnowledge tab-delimited text export.

    Parameters
    ----------
    filepath : str
        Path to the BIOPAC .txt file.

    Returns
    -------
    df : pd.DataFrame
        Numeric data with columns milliSec, CH1 … CH47, time_s.
    meta : dict
        Recording metadata (filename, recording_time, fs).
    """
    meta = {}
    with open(filepath, "r") as f:
        lines = f.readlines()
    meta["filename"] = lines[0].strip()
    meta["recording_time"] = lines[2].strip().replace("Recording on: ", "")
    meta["fs"] = 1000
    col_names = [
        "milliSec", "CH1", "CH2", "CH3", "CH40", "CH41", "CH42",
        "CH43", "CH44", "CH45", "CH46", "CH47", "_extra",
    ]
    df = pd.read_csv(
        filepath, sep="\t", skiprows=29, header=None,
        names=col_names, on_bad_lines="skip",
    )
    df = df.drop(columns=["_extra"], errors="ignore")
    df = df.dropna(subset=["CH40"])
    df["time_s"] = df["milliSec"] / 1000.0
    return df, meta


def parse_belt(filepath):
    """Parse Baby Belt BLE-streamed CSV.

    Resamples the jittered BLE stream onto a uniform 100 Hz grid
    via linear interpolation.

    Parameters
    ----------
    filepath : str
        Path to the belt .csv file.

    Returns
    -------
    df : pd.DataFrame
        Raw belt DataFrame.
    ecg_interp : np.ndarray
        Uniformly resampled ECG (ADC counts).
    t_uniform : np.ndarray
        Uniform time vector (seconds).
    fs_nominal : int
        Nominal sampling rate (100 Hz).
    """
    df = pd.read_csv(filepath)
    df["time_s"] = (df["Tx_ms"] - df["Tx_ms"].iloc[0]) / 1000.0
    df["seq_gap"] = df["Seq"].diff() > 1
    fs_nominal = 100
    t_uniform = np.arange(0, df["time_s"].iloc[-1], 1 / fs_nominal)
    ecg_interp = np.interp(t_uniform, df["time_s"].values, df["ECG"].values)
    return df, ecg_interp, t_uniform, fs_nominal


# -----------------------------------------------------------------
# Filters
# -----------------------------------------------------------------

def bandpass_filter(signal, fs, lowcut=0.5, highcut=40.0, order=5):
    """Zero-phase Butterworth bandpass filter.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : int
        Sampling frequency (Hz).
    lowcut, highcut : float
        Passband edges (Hz).
    order : int
        Filter order.

    Returns
    -------
    np.ndarray
        Filtered signal.
    """
    sos = butter(order, [lowcut, highcut], btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, signal)


def notch_filter(signal, fs, freq=60.0, quality=30.0):
    """Zero-phase IIR notch filter for powerline removal.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : int
        Sampling frequency (Hz).
    freq : float
        Notch centre frequency (Hz).
    quality : float
        Quality factor.

    Returns
    -------
    np.ndarray
        Filtered signal.
    """
    nyquist = fs / 2.0
    if freq >= 0.9 * nyquist:
        print(f'  [notch_filter] Skipping {freq:.0f} Hz notch — '
              f'too close to Nyquist ({nyquist:.0f} Hz) for fs={fs:.0f} Hz')
        return signal
    b, a = iirnotch(freq, quality, fs)
    sos = tf2sos(b, a)
    return sosfiltfilt(sos, signal)


def remove_baseline(signal, fs):
    """Remove baseline wander via cascaded median filters.

    Parameters
    ----------
    signal : np.ndarray
        Input signal.
    fs : int
        Sampling frequency (Hz).

    Returns
    -------
    np.ndarray
        Baseline-corrected signal.
    """
    win1 = int(0.2 * fs) | 1
    win2 = int(0.6 * fs) | 1
    baseline = medfilt(medfilt(signal, win1), win2)
    return signal - baseline


# -----------------------------------------------------------------
# Evaluation
# -----------------------------------------------------------------

def evaluate_rpeaks(ref_peaks, test_peaks, fs, tolerance_ms=50):
    """Evaluate R-peak detection accuracy against a reference.

    Uses a greedy nearest-neighbour matcher with a tolerance window.

    Parameters
    ----------
    ref_peaks : np.ndarray
        Reference R-peak sample indices.
    test_peaks : np.ndarray
        Detected R-peak sample indices.
    fs : int
        Sampling frequency (Hz).
    tolerance_ms : float
        Maximum allowed deviation in milliseconds.

    Returns
    -------
    dict
        Keys: TP, FP, FN, Se, PPV, F1.
    """
    if len(ref_peaks) == 0 or len(test_peaks) == 0:
        return dict(TP=0, FP=len(test_peaks), FN=len(ref_peaks),
                    Se=0.0, PPV=0.0, F1=0.0)

    tol = int(tolerance_ms * fs / 1000)
    tp, fp = 0, 0
    matched = set()

    for p in test_peaks:
        diffs = np.abs(ref_peaks - p)
        idx = np.argmin(diffs)
        if diffs[idx] <= tol and idx not in matched:
            tp += 1
            matched.add(idx)
        else:
            fp += 1

    fn = len(ref_peaks) - len(matched)
    se = tp / (tp + fn + 1e-9)
    ppv = tp / (tp + fp + 1e-9)
    f1 = 2 * se * ppv / (se + ppv + 1e-9)
    return dict(TP=tp, FP=fp, FN=fn,
                Se=round(se, 4), PPV=round(ppv, 4), F1=round(f1, 4))


# ===================== CareWear BIOPAC Parser =====================

def parse_carewear_biopac(filepath, ecg_col=None):
    """Parse CareWear BIOPAC AcqKnowledge exported .txt file.

    Reads a variable-length header, auto-detects sampling rate
    and column layout, returns (df, meta) matching parse_biopac().

    Parameters
    ----------
    filepath : str
        Path to the CareWear BIOPAC .txt export file.
    ecg_col : str or None
        Column to use as primary ECG (default: CAREWEAR_BIOPAC_ECG_COL).

    Returns
    -------
    df : pd.DataFrame
        Numeric data with columns milliSec, CH5, CH9, CH40, etc., time_s.
    meta : dict
        Recording metadata (filename, recording_time, fs).
    """
    if ecg_col is None:
        ecg_col = CAREWEAR_BIOPAC_ECG_COL

    meta = {}
    with open(filepath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    meta["filename"] = lines[0].strip()
    sample_rate_ms = None
    header_idx = None

    for idx, line in enumerate(lines):
        stripped = line.strip()
        if not stripped:
            continue
        if "msec/sample" in stripped and sample_rate_ms is None:
            sample_rate_ms = float(stripped.split()[0])
        if stripped.startswith("Recording on:"):
            meta["recording_time"] = stripped.split(": ", 1)[1]
        if stripped.startswith("milliSec"):
            header_idx = idx
            break

    if sample_rate_ms is None:
        raise ValueError("Could not find sample rate (msec/sample) in header.")
    if header_idx is None:
        raise ValueError("Could not find data header row starting with milliSec.")

    meta["fs"] = int(round(1000.0 / sample_rate_ms))

    col_names = [c.strip() for c in lines[header_idx].strip().split("\t") if c.strip()]

    from io import StringIO
    data_text = "".join(lines[header_idx + 2 :])
    df = pd.read_csv(StringIO(data_text), sep="\t", header=None, on_bad_lines="skip")
    df = df.dropna(how="all", axis=1)
    df.columns = col_names[: len(df.columns)]

    if ecg_col not in df.columns:
        raise ValueError(f"ECG column '{ecg_col}' not found. Available: {list(df.columns)}")

    df = df.dropna(subset=[ecg_col])
    df["time_s"] = df["milliSec"] / 1000.0
    return df, meta



# ===================== CareWear Belt Parser =====================

def parse_carewear_belt(filepath, ecg_scale_fn=None):
    """Parse CareWear belt data file (CSV with Unix epoch timestamps).

    Reads the CSV, converts epoch timestamps to relative seconds,
    z-score normalises Channel 4 (ECG), and resamples onto a uniform
    grid.  Returns (df, ecg_interp, t_uniform, fs_nominal) matching
    parse_belt().

    Parameters
    ----------
    filepath : str
        Path to the CareWear belt data file.
    ecg_scale_fn : callable or None
        Optional scaling applied to raw ECG before normalisation.
        Defaults to CAREWEAR_BELT_ECG_SCALE_FN.

    Returns
    -------
    df : pd.DataFrame
        Raw DataFrame with added time_s and seq_gap columns.
    ecg_interp : np.ndarray
        ECG resampled to a uniform grid (z-score normalised).
    t_uniform : np.ndarray
        Uniform time axis in seconds.
    fs_nominal : int
        Estimated sampling rate in Hz.
    """
    if ecg_scale_fn is None:
        ecg_scale_fn = CAREWEAR_BELT_ECG_SCALE_FN

    df = pd.read_csv(filepath)
    ts = df["timestamp"].values.astype(np.float64)
    df["time_s"] = (ts - ts[0]) / 1000.0

    ecg_raw = df["Channel 4"].values.astype(np.float64)
    if ecg_scale_fn is not None:
        ecg_raw = ecg_scale_fn(ecg_raw)

    ecg_mean, ecg_std = np.mean(ecg_raw), np.std(ecg_raw)
    ecg_norm = (ecg_raw - ecg_mean) / ecg_std if ecg_std > 0 else ecg_raw - ecg_mean

    dt_ms = np.median(np.diff(ts))
    fs_nominal = int(round(1000.0 / dt_ms))

    t_uniform = np.arange(0, df["time_s"].iloc[-1], 1.0 / fs_nominal)
    ecg_interp = np.interp(t_uniform, df["time_s"].values, ecg_norm)
    df["seq_gap"] = False

    return df, ecg_interp, t_uniform, fs_nominal



In [ ]:
# -----------------------------------------------------------------
# Load data
# -----------------------------------------------------------------
if DATASET_FORMAT == "CAREWEAR":
    biopac_df, biopac_meta = parse_carewear_biopac(CAREWEAR_BIOPAC_FILE_PATH)
    belt_df, belt_ecg_raw, belt_time, belt_fs = parse_carewear_belt(CAREWEAR_BELT_FILE_PATH)
else:
    biopac_df, biopac_meta = parse_biopac(BIOPAC_FILE_PATH)
    belt_df, belt_ecg_raw, belt_time, belt_fs = parse_belt(BELT_FILE_PATH)

biopac_fs = biopac_meta["fs"]

print(f"BIOPAC  : {len(biopac_df)} samples @ {biopac_fs} Hz  "
      f"({biopac_df['time_s'].iloc[-1]:.1f} s)")
print(f"Belt    : {len(belt_ecg_raw)} samples @ {belt_fs} Hz  "
      f"({belt_time[-1]:.1f} s)")

# -----------------------------------------------------------------
# BIOPAC CH40 + belt: full pipeline (notch -> bandpass -> baseline)
# CH1 = sync/aux only — not used as ECG.
# -----------------------------------------------------------------
biopac_mask = biopac_df["time_s"] >= 0
if DATASET_FORMAT == "CAREWEAR":
    _ecg_col = CAREWEAR_BIOPAC_ECG_COL
else:
    _ecg_col = "CH40"
biopac_ecg_ch40 = biopac_df.loc[biopac_mask, _ecg_col].values.astype(np.float64)

biopac_ecg_clean = remove_baseline(
    bandpass_filter(
        notch_filter(biopac_ecg_ch40, biopac_fs, 60.0),
        biopac_fs, 0.5, 40.0,
    ),
    biopac_fs,
)

belt_ecg_clean = remove_baseline(
    bandpass_filter(
        notch_filter(belt_ecg_raw, belt_fs, 60.0),
        belt_fs, 0.5, 40.0,
    ),
    belt_fs,
)

# -----------------------------------------------------------------
# Reference R-peaks: NeuroKit2 ecg_process on the pipeline signal above
ref_signals, ref_info = nk.ecg_process(biopac_ecg_clean, sampling_rate=biopac_fs)
ref_peaks = np.where(ref_signals["ECG_R_Peaks"].values == 1)[0]

biopac_time_arr = biopac_df.loc[biopac_mask, "time_s"].values

print(f"\nReference peaks (NeuroKit2 on CH40): {len(ref_peaks)}")
print(f"Recording duration: {biopac_time_arr[-1]:.1f} s")

## 5. RPNet Model Definition (IncRes-UNet)

The full model is copied inline from `reference_codes/RPNet/network.py` to
keep this notebook self-contained. The architecture has ~18.7M parameters.

In [ ]:
if TORCH_AVAILABLE:

    class IncResBlock(nn.Module):
        """Inception-Residual block with four parallel 1-D convolutions.

        Each branch uses a different kernel size (convsize, +2, +4, +6)
        to capture multi-scale temporal features. Outputs are concatenated
        and added to a 1x1 residual projection of the input.

        Parameters
        ----------
        inplanes : int
            Number of input channels.
        planes : int
            Number of output channels (split equally across 4 branches).
        convstr : int
            Convolution stride.
        convsize : int
            Base kernel size for the first branch.
        convpadding : int
            Base padding for the first branch.
        """

        def __init__(self, inplanes, planes, convstr=1, convsize=15, convpadding=7):
            super(IncResBlock, self).__init__()
            self.Inputconv1x1 = nn.Conv1d(
                inplanes, planes, kernel_size=1, stride=1, bias=False
            )
            self.conv1_1 = nn.Sequential(
                nn.Conv1d(
                    in_channels=inplanes, out_channels=planes // 4,
                    kernel_size=convsize, stride=convstr, padding=convpadding,
                ),
                nn.BatchNorm1d(planes // 4),
            )
            self.conv1_2 = nn.Sequential(
                nn.Conv1d(inplanes, planes // 4, kernel_size=1,
                          stride=convstr, padding=0, bias=False),
                nn.BatchNorm1d(planes // 4),
                nn.LeakyReLU(0.2),
                nn.Conv1d(
                    in_channels=planes // 4, out_channels=planes // 4,
                    kernel_size=convsize + 2, stride=convstr,
                    padding=convpadding + 1,
                ),
                nn.BatchNorm1d(planes // 4),
            )
            self.conv1_3 = nn.Sequential(
                nn.Conv1d(inplanes, planes // 4, kernel_size=1,
                          stride=convstr, padding=0, bias=False),
                nn.BatchNorm1d(planes // 4),
                nn.LeakyReLU(0.2),
                nn.Conv1d(
                    in_channels=planes // 4, out_channels=planes // 4,
                    kernel_size=convsize + 4, stride=convstr,
                    padding=convpadding + 2,
                ),
                nn.BatchNorm1d(planes // 4),
            )
            self.conv1_4 = nn.Sequential(
                nn.Conv1d(inplanes, planes // 4, kernel_size=1,
                          stride=convstr, padding=0, bias=False),
                nn.BatchNorm1d(planes // 4),
                nn.LeakyReLU(0.2),
                nn.Conv1d(
                    in_channels=planes // 4, out_channels=planes // 4,
                    kernel_size=convsize + 6, stride=convstr,
                    padding=convpadding + 3,
                ),
                nn.BatchNorm1d(planes // 4),
            )
            self.relu = nn.ReLU()

        def forward(self, x):
            residual = self.Inputconv1x1(x)
            c1 = self.conv1_1(x)
            c2 = self.conv1_2(x)
            c3 = self.conv1_3(x)
            c4 = self.conv1_4(x)
            out = torch.cat([c1, c2, c3, c4], 1)
            out += residual
            out = self.relu(out)
            return out


    class IncUNet(nn.Module):
        """Inception-Residual U-Net for ECG R-peak distance transform prediction.

        Encoder-decoder architecture with skip connections. Each encoder/decoder
        stage uses IncResBlock for multi-scale feature extraction.

        Parameters
        ----------
        in_shape : tuple of (int, int, int)
            (channels, height, width) — for 1-D ECG: (1, 1, 5000).
        """

        def __init__(self, in_shape):
            super(IncUNet, self).__init__()
            in_channels, height, width = in_shape

            # ---- Encoder ----
            self.e1 = nn.Sequential(
                nn.Conv1d(in_channels, 64, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(64),
                nn.LeakyReLU(0.2),
                IncResBlock(64, 64),
            )
            self.e2 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.Conv1d(64, 128, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(128),
                IncResBlock(128, 128),
            )
            self.e2add = nn.Sequential(
                nn.Conv1d(128, 128, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(128),
            )
            self.e3 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.Conv1d(128, 128, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(128),
                nn.LeakyReLU(0.2),
                nn.Conv1d(128, 256, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(256),
                IncResBlock(256, 256),
            )
            self.e4 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.Conv1d(256, 256, kernel_size=4, stride=1, padding=1),
                nn.BatchNorm1d(256),
                nn.LeakyReLU(0.2),
                nn.Conv1d(256, 512, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(512),
                IncResBlock(512, 512),
            )
            self.e4add = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.Conv1d(512, 512, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(512),
            )
            self.e5 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.Conv1d(512, 512, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(512),
                nn.LeakyReLU(0.2),
                nn.Conv1d(512, 512, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(512),
                IncResBlock(512, 512),
            )
            self.e6 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.Conv1d(512, 512, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(512),
                nn.LeakyReLU(0.2),
                nn.Conv1d(512, 512, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(512),
                IncResBlock(512, 512),
            )
            self.e6add = nn.Sequential(
                nn.Conv1d(512, 512, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(512),
            )
            self.e7 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.Conv1d(512, 512, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(512),
                nn.LeakyReLU(0.2),
                nn.Conv1d(512, 512, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(512),
                IncResBlock(512, 512),
            )
            self.e8 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.Conv1d(512, 512, kernel_size=4, stride=1, padding=1),
                nn.BatchNorm1d(512),
                nn.LeakyReLU(0.2),
                nn.Conv1d(512, 512, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(512),
            )

            # ---- Decoder ----
            self.d1 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(512, 512, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(512),
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(512, 512, kernel_size=4, stride=1, padding=1),
                nn.BatchNorm1d(512),
                IncResBlock(512, 512),
            )
            self.d2 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(1024, 512, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(512),
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(512, 512, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(512),
                IncResBlock(512, 512),
            )
            self.d3 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(1024, 512, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(512),
                nn.Dropout(p=0.5),
                IncResBlock(512, 512),
            )
            self.d4 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(1024, 512, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(512),
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(512, 512, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(512),
                IncResBlock(512, 512),
            )
            self.d5 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(1024, 512, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(512),
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(512, 512, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(512),
                IncResBlock(512, 512),
            )
            self.d6 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(1024, 512, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(512),
                IncResBlock(512, 512),
            )
            self.d7 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(1024, 256, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(256),
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(256, 256, kernel_size=4, stride=1, padding=1),
                nn.BatchNorm1d(256),
                IncResBlock(256, 256),
            )
            self.d8 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(512, 128, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(128),
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(128, 128, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(128),
            )
            self.d9 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(256, 128, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm1d(128),
            )
            self.d10 = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(256, 64, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm1d(64),
            )
            self.out_l = nn.Sequential(
                nn.LeakyReLU(0.2),
                nn.ConvTranspose1d(128, in_channels, kernel_size=4, stride=2,
                                   padding=1),
            )

        def forward(self, x):
            en1 = self.e1(x)
            en2 = self.e2(en1)
            en2add = self.e2add(en2)
            en3 = self.e3(en2add)
            en4 = self.e4(en3)
            en4add = self.e4add(en4)
            en5 = self.e5(en4add)
            en6 = self.e6(en5)
            en6add = self.e6add(en6)
            en7 = self.e7(en6add)
            en8 = self.e8(en7)

            de1_ = self.d1(en8)
            de1 = torch.cat([en7, de1_], 1)
            de2_ = self.d2(de1)
            de2 = torch.cat([en6add, de2_], 1)
            de3_ = self.d3(de2)
            de3 = torch.cat([en6, de3_], 1)
            de4_ = self.d4(de3)
            de4 = torch.cat([en5, de4_], 1)
            de5_ = self.d5(de4)
            de5 = torch.cat([en4add, de5_], 1)
            de6_ = self.d6(de5)
            de6 = torch.cat([en4, de6_], 1)
            de7_ = self.d7(de6)
            de7 = torch.cat([en3, de7_], 1)
            de8_ = self.d8(de7)
            de8 = torch.cat([en2add, de8_], 1)
            de9_ = self.d9(de8)
            de9 = torch.cat([en2, de9_], 1)
            de10_ = self.d10(de9)
            de10 = torch.cat([en1, de10_], 1)
            out = self.out_l(de10)

            return out


    total_params = sum(
        p.numel() for p in IncUNet(in_shape=(1, 1, 5000)).parameters()
    )
    print(f"IncUNet parameters: {total_params:,} ({total_params / 1e6:.1f}M)")

else:
    print("Skipping model definition — PyTorch not available.")

## 6. Load Pre-Trained Weights

The model file should be placed at the path configured in `RPNET_WEIGHTS`.
If the file is not found, the notebook will use an untrained model — the
inference pipeline will still execute, but the output distance map will
be meaningless.

In [ ]:
def load_rpnet_model(weights_path, device="cpu"):
    """Load RPNet (IncUNet) model with pre-trained weights.

    Parameters
    ----------
    weights_path : str
        Path to .pt or .pth weights file.
    device : str
        'cpu' or 'cuda'.

    Returns
    -------
    model : IncUNet or None
        Loaded model in eval mode, or None if PyTorch unavailable.
    """
    if not TORCH_AVAILABLE:
        print("PyTorch not available — cannot load model.")
        return None

    C, H, W = 1, 1, 5000
    model = IncUNet(in_shape=(C, H, W))

    if os.path.exists(weights_path):
        try:
            state_dict = torch.load(
                weights_path, map_location=device, weights_only=False
            )
            if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
                model.load_state_dict(state_dict["model_state_dict"])
            else:
                model.load_state_dict(state_dict)
            print(f"RPNet weights loaded from: {weights_path}")
        except Exception as e:
            print(f"Failed to load weights: {e}")
            print("Using untrained model for demonstration.")
    else:
        print(f"WARNING: Weights file not found at {weights_path}")
        print("Download from: https://drive.google.com/file/d/"
              "19xN7pZsALb09bxWjrSKdAlJmRqYL0M0g/view")
        print("Using untrained model for demonstration.")

    model.to(device)
    model.eval()
    return model


rpnet_model = load_rpnet_model(RPNET_WEIGHTS, device=DEVICE)

## 7. Sampling Rate Conversion

Both signals must be resampled to 500 Hz before feeding them to RPNet.
`resample_poly` applies an anti-aliasing FIR filter internally.

In [ ]:
from math import gcd as _gcd

FS_RPNET = 500
bp_fs_int = int(round(biopac_fs))
bl_fs_int = int(round(belt_fs))

up_bp = FS_RPNET // _gcd(bp_fs_int, FS_RPNET)
dn_bp = bp_fs_int // _gcd(bp_fs_int, FS_RPNET)
biopac_ecg_500 = resample_poly(biopac_ecg_clean, up=up_bp, down=dn_bp)
print(f"BIOPAC: {len(biopac_ecg_clean):,} samples @ {bp_fs_int} Hz "
      f"→ {len(biopac_ecg_500):,} @ {FS_RPNET} Hz  (up={up_bp}, down={dn_bp})")

up_bl = FS_RPNET // _gcd(bl_fs_int, FS_RPNET)
dn_bl = bl_fs_int // _gcd(bl_fs_int, FS_RPNET)
belt_ecg_500 = resample_poly(belt_ecg_clean, up=up_bl, down=dn_bl)
print(f"Belt:   {len(belt_ecg_clean):,} samples @ {bl_fs_int} Hz "
      f"→ {len(belt_ecg_500):,} @ {FS_RPNET} Hz  (up={up_bl}, down={dn_bl})")

## 8. Windowed RPNet Inference

RPNet expects 10-second windows (5000 samples at 500 Hz). For recordings
longer than 10 s we slide a window with 50% overlap and average the
predictions in overlapping regions. R-peaks are extracted as local minima
of the distance transform map.

In [ ]:
def rpnet_inference_windowed(signal_500hz, model, device="cpu",
                              window_samples=5000, stride=2500):
    """Run RPNet on a full signal using overlapping 10-second windows.

    Parameters
    ----------
    signal_500hz : np.ndarray
        ECG signal resampled to 500 Hz.
    model : IncUNet
        Loaded RPNet model.
    device : str
        'cpu' or 'cuda'.
    window_samples : int
        Window size (5000 = 10 s at 500 Hz).
    stride : int
        Stride between windows (2500 = 50% overlap).

    Returns
    -------
    distance_map : np.ndarray
        Stitched distance transform map (same length as input).
    r_peaks : np.ndarray
        Detected R-peak indices in the 500 Hz domain.
    """
    n = len(signal_500hz)

    pad_len = stride - (n % stride) if n % stride != 0 else 0
    signal_padded = np.pad(signal_500hz, (0, pad_len), mode="reflect")

    if len(signal_padded) < window_samples:
        signal_padded = np.pad(
            signal_padded,
            (0, window_samples - len(signal_padded)),
            mode="reflect",
        )

    n_padded = len(signal_padded)
    distance_map = np.zeros(n_padded)
    weight_map = np.zeros(n_padded)

    n_windows = max(1, (n_padded - window_samples) // stride + 1)

    model.eval()
    with torch.no_grad():
        for i in tqdm(range(n_windows), desc="RPNet inference"):
            start = i * stride
            end = start + window_samples
            if end > n_padded:
                break

            window = signal_padded[start:end]

            mu = np.mean(window)
            sigma = np.std(window) + 1e-8
            window_norm = (window - mu) / sigma

            x = (
                torch.from_numpy(window_norm)
                .unsqueeze(0)
                .unsqueeze(0)
                .float()
                .to(device)
            )

            pred = model(x)
            dt = pred.squeeze().cpu().numpy()

            distance_map[start:end] += dt
            weight_map[start:end] += 1

    weight_map = np.maximum(weight_map, 1)
    distance_map = distance_map / weight_map

    distance_map = distance_map[:n]

    inv_dt = -distance_map
    r_peaks, _ = find_peaks(inv_dt, distance=int(0.3 * 500), prominence=2.0)

    return distance_map, r_peaks

## 9. Run Inference on Both Signals

We time each inference run to compare computational cost against classical
detectors.

In [ ]:
if rpnet_model is not None:
    # --- BIOPAC ---
    t0 = time.perf_counter()
    biopac_dt_map, biopac_rpnet_peaks = rpnet_inference_windowed(
        biopac_ecg_500, rpnet_model, device=DEVICE
    )
    biopac_rpnet_time = (time.perf_counter() - t0) * 1000

    print(f"\nBIOPAC RPNet: {len(biopac_rpnet_peaks)} peaks detected "
          f"in {biopac_rpnet_time:.0f} ms")

    # --- Belt ---
    t0 = time.perf_counter()
    belt_dt_map, belt_rpnet_peaks = rpnet_inference_windowed(
        belt_ecg_500, rpnet_model, device=DEVICE
    )
    belt_rpnet_time = (time.perf_counter() - t0) * 1000

    print(f"Belt RPNet:   {len(belt_rpnet_peaks)} peaks detected "
          f"in {belt_rpnet_time:.0f} ms")

else:
    print("RPNet model not available — skipping inference.")
    biopac_dt_map = np.zeros(len(biopac_ecg_500))
    biopac_rpnet_peaks = np.array([], dtype=int)
    belt_dt_map = np.zeros(len(belt_ecg_500))
    belt_rpnet_peaks = np.array([], dtype=int)
    biopac_rpnet_time = 0
    belt_rpnet_time = 0

## 10. Distance Transform Visualization

Three-panel plot showing the raw ECG (at 500 Hz), the predicted distance
transform map, and R-peaks overlaid on the ECG.

In [ ]:
def plot_distance_transform(ecg_500, dt_map, peaks_500, fs=500,
                            title_prefix="", t_start=0, t_end=20,
                            save_path=None):
    """Plot ECG, distance transform, and detected peaks in three panels.

    Parameters
    ----------
    ecg_500 : np.ndarray
        ECG signal at 500 Hz.
    dt_map : np.ndarray
        Distance transform map from RPNet.
    peaks_500 : np.ndarray
        R-peak indices in the 500 Hz domain.
    fs : int
        Sampling rate (500).
    title_prefix : str
        Prefix for the figure title.
    t_start, t_end : float
        Time window to display (seconds).
    save_path : str or None
        If provided, save figure to this path.
    """
    idx_start = int(t_start * fs)
    idx_end = min(int(t_end * fs), len(ecg_500))
    t = np.arange(idx_start, idx_end) / fs
    ecg_seg = ecg_500[idx_start:idx_end]
    dt_seg = dt_map[idx_start:idx_end]

    peaks_in_win = peaks_500[
        (peaks_500 >= idx_start) & (peaks_500 < idx_end)
    ]

    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

    axes[0].plot(t, ecg_seg, color="#1976D2", linewidth=0.7)
    axes[0].set_ylabel("Amplitude")
    axes[0].set_title(f"{title_prefix} — ECG Signal (500 Hz)")

    axes[1].plot(t, dt_seg, color="#7B1FA2", linewidth=0.8)
    axes[1].fill_between(t, dt_seg, alpha=0.15, color="#CE93D8")
    axes[1].set_ylabel("Distance Transform")
    axes[1].set_title(f"{title_prefix} — RPNet Distance Transform Output")

    axes[2].plot(t, ecg_seg, color="#1976D2", linewidth=0.7, label="ECG")
    if len(peaks_in_win) > 0:
        axes[2].plot(
            peaks_in_win / fs, ecg_500[peaks_in_win],
            "rv", markersize=10, label="RPNet R-peaks",
        )
    axes[2].set_ylabel("Amplitude")
    axes[2].set_xlabel("Time (s)")
    axes[2].set_title(f"{title_prefix} — Detected R-Peaks")
    axes[2].legend(loc="upper right")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {save_path}")
    plt.show()


plot_distance_transform(
    biopac_ecg_500, biopac_dt_map, biopac_rpnet_peaks,
    title_prefix="BIOPAC", t_start=0, t_end=20,
    save_path=os.path.join(OUTPUT_DIR, "biopac_rpnet_distance_transform.png"),
)

plot_distance_transform(
    belt_ecg_500, belt_dt_map, belt_rpnet_peaks,
    title_prefix="Belt", t_start=0, t_end=20,
    save_path=os.path.join(OUTPUT_DIR, "belt_rpnet_distance_transform.png"),
)

## 11. Convert Peaks Back to Original Sampling Rates

RPNet detected peaks in the 500 Hz domain. To compare with classical
detectors (which operate at native sampling rates) we must map the peak
indices back.

In [ ]:
# Convert 500 Hz peak indices → original fs indices
biopac_rpnet_peaks_original = np.round(
    biopac_rpnet_peaks * (1000 / 500)
).astype(int)

belt_rpnet_peaks_original = np.round(
    belt_rpnet_peaks * (100 / 500)
).astype(int)

# Clip to valid range
biopac_rpnet_peaks_original = biopac_rpnet_peaks_original[
    biopac_rpnet_peaks_original < len(biopac_ecg_clean)
]
belt_rpnet_peaks_original = belt_rpnet_peaks_original[
    belt_rpnet_peaks_original < len(belt_ecg_clean)
]

print(f"BIOPAC RPNet peaks mapped to 1000 Hz: {len(biopac_rpnet_peaks_original)}")
print(f"Belt RPNet peaks mapped to 100 Hz:    {len(belt_rpnet_peaks_original)}")

## 12. Evaluate RPNet Against BIOPAC Reference

We use the same beat-level evaluation (±50 ms tolerance) from NB04.

In [ ]:
# --- BIOPAC CH40 (pipeline): RPNet vs reference ---
biopac_rpnet_eval = evaluate_rpeaks(
    ref_peaks, biopac_rpnet_peaks_original, biopac_fs
)

print("=== BIOPAC CH40 (pipeline) — RPNet vs Reference (CH40) ===")
print(f"  Reference peaks : {len(ref_peaks)}")
print(f"  RPNet detected  : {len(biopac_rpnet_peaks_original)}")
print(f"  TP={biopac_rpnet_eval['TP']}  "
      f"FP={biopac_rpnet_eval['FP']}  "
      f"FN={biopac_rpnet_eval['FN']}")
print(f"  Sensitivity : {biopac_rpnet_eval['Se']:.4f}")
print(f"  PPV         : {biopac_rpnet_eval['PPV']:.4f}")
print(f"  F1          : {biopac_rpnet_eval['F1']:.4f}")

# --- Belt: RPNet vs BIOPAC-mapped reference ---
ref_peak_times = biopac_time_arr[ref_peaks]
belt_ref_indices = np.array([
    np.argmin(np.abs(belt_time - t)) for t in ref_peak_times
    if t <= belt_time[-1]
])

belt_rpnet_eval = evaluate_rpeaks(
    belt_ref_indices, belt_rpnet_peaks_original, belt_fs
)

print(f"\n=== Belt — RPNet vs BIOPAC-Mapped Reference ===")
print(f"  Reference peaks : {len(belt_ref_indices)}")
print(f"  RPNet detected  : {len(belt_rpnet_peaks_original)}")
print(f"  TP={belt_rpnet_eval['TP']}  "
      f"FP={belt_rpnet_eval['FP']}  "
      f"FN={belt_rpnet_eval['FN']}")
print(f"  Sensitivity : {belt_rpnet_eval['Se']:.4f}")
print(f"  PPV         : {belt_rpnet_eval['PPV']:.4f}")
print(f"  F1          : {belt_rpnet_eval['F1']:.4f}")

## 13. Visual Comparison — RPNet vs Classical Detectors

Side-by-side overlay on a 10-second window showing RPNet peaks alongside
Pan-Tompkins and the BIOPAC reference.

In [ ]:
# Run Pan-Tompkins for comparison
try:
    _, pt_info_biopac = nk.ecg_peaks(
        biopac_ecg_clean, sampling_rate=biopac_fs,
        method="pantompkins1985",
    )
    pt_peaks_biopac = pt_info_biopac["ECG_R_Peaks"]
except Exception:
    pt_peaks_biopac = np.array([])

try:
    _, pt_info_belt = nk.ecg_peaks(
        belt_ecg_clean, sampling_rate=belt_fs,
        method="pantompkins1985",
    )
    pt_peaks_belt = pt_info_belt["ECG_R_Peaks"]
except Exception:
    pt_peaks_belt = np.array([])

# --- Plot ---
WINDOW_START_S = 5
WINDOW_END_S = 15

fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# -- BIOPAC --
idx_s_b = int(WINDOW_START_S * biopac_fs)
idx_e_b = int(WINDOW_END_S * biopac_fs)
t_biopac = biopac_time_arr[idx_s_b:idx_e_b]
ecg_biopac_seg = biopac_ecg_clean[idx_s_b:idx_e_b]

axes[0].plot(t_biopac, ecg_biopac_seg, color="#90A4AE", linewidth=0.7,
             label="BIOPAC (CH40, pipeline)")

ref_in_win = ref_peaks[(ref_peaks >= idx_s_b) & (ref_peaks < idx_e_b)]
axes[0].plot(biopac_time_arr[ref_in_win], biopac_ecg_clean[ref_in_win],
             "b^", markersize=12, zorder=10, label="Reference (CH40)")

rpnet_in_win = biopac_rpnet_peaks_original[
    (biopac_rpnet_peaks_original >= idx_s_b)
    & (biopac_rpnet_peaks_original < idx_e_b)
]
if len(rpnet_in_win) > 0:
    axes[0].plot(
        biopac_time_arr[rpnet_in_win], biopac_ecg_clean[rpnet_in_win],
        "ro", markersize=9, zorder=9, label="RPNet",
    )

pt_in_win = pt_peaks_biopac[
    (pt_peaks_biopac >= idx_s_b) & (pt_peaks_biopac < idx_e_b)
]
if len(pt_in_win) > 0:
    axes[0].plot(
        biopac_time_arr[pt_in_win], biopac_ecg_clean[pt_in_win],
        "*", color="#FF9800", markersize=11, zorder=8, label="Pan-Tompkins",
    )

axes[0].set_title("BIOPAC — RPNet vs Pan-Tompkins vs Reference")
axes[0].set_ylabel("Amplitude")
axes[0].legend(loc="upper right")

# -- Belt --
idx_s_belt = int(WINDOW_START_S * belt_fs)
idx_e_belt = int(WINDOW_END_S * belt_fs)
t_belt_seg = belt_time[idx_s_belt:idx_e_belt]
ecg_belt_seg = belt_ecg_clean[idx_s_belt:idx_e_belt]

axes[1].plot(t_belt_seg, ecg_belt_seg, color="#90A4AE", linewidth=0.7,
             label="Belt ECG")

rpnet_belt_in = belt_rpnet_peaks_original[
    (belt_rpnet_peaks_original >= idx_s_belt)
    & (belt_rpnet_peaks_original < idx_e_belt)
]
if len(rpnet_belt_in) > 0:
    axes[1].plot(
        belt_time[rpnet_belt_in], belt_ecg_clean[rpnet_belt_in],
        "ro", markersize=9, zorder=9, label="RPNet",
    )

pt_belt_in = pt_peaks_belt[
    (pt_peaks_belt >= idx_s_belt) & (pt_peaks_belt < idx_e_belt)
]
if len(pt_belt_in) > 0:
    axes[1].plot(
        belt_time[pt_belt_in], belt_ecg_clean[pt_belt_in],
        "*", color="#FF9800", markersize=11, zorder=8, label="Pan-Tompkins",
    )

axes[1].set_title("Belt — RPNet vs Pan-Tompkins")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Amplitude")
axes[1].legend(loc="upper right")

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "rpnet_vs_classical_comparison.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved: {save_path}")
plt.show()

## 14. When RPNet Helps vs Hurts

### Where deep learning shines

| Scenario | Why RPNet helps |
|---|---|
| **Noisy segments** | Learned features are more robust to additive noise than threshold-based rules |
| **Morphology distortion** | Textile electrodes produce wide/split QRS that confuse derivative detectors |
| **Motion artifacts** | Short-duration artifacts that classical detectors misinterpret as QRS |
| **Low SNR** | Distance transform provides a smooth regression target even when peaks are ambiguous |

### Where it struggles

| Scenario | Why |
|---|---|
| **Wrong sampling rate** | Input must be exactly 500 Hz — always resample first |
| **Out-of-distribution morphology** | Trained on MIT-BIH (adult, standard leads); pediatric/fetal ECG may not generalize |
| **Very long artifacts** | Entire 10-s windows dominated by artifact produce unreliable distance maps |
| **Computational cost** | ~18.7M parameters; significantly slower than classical methods on CPU |

### Generalization: MIT-BIH → Textile ECG

RPNet was trained on the MIT-BIH Arrhythmia Database — clinical-grade
recordings with standard Ag/AgCl electrodes. Textile ECG from wearable
belts has systematically different characteristics:

- Higher baseline wander (body movement, loose contact)
- Lower QRS amplitude and different morphology
- More frequent motion artifacts

The distance transform formulation provides some robustness (the model
learns "where peaks are" rather than "what peaks look like"), but
fine-tuning on textile ECG data would likely improve performance
significantly.

## 15. Segment-Level Analysis: Signal Quality vs Detection F1

We compute a kurtosis-based signal quality index (SQI) for each window
and scatter it against the windowed F1 score. This reveals whether RPNet
maintains higher accuracy than classical methods in low-quality segments.

In [ ]:
def compute_kurtosis_sqi(signal, fs, window_s=5, stride_s=2.5):
    """Compute kurtosis-based signal quality index per window.

    Higher kurtosis (peaky distribution) suggests clean QRS; low kurtosis
    suggests noise or artifact dominance.

    Parameters
    ----------
    signal : np.ndarray
        ECG signal.
    fs : int
        Sampling frequency.
    window_s : float
        Window duration in seconds.
    stride_s : float
        Stride in seconds.

    Returns
    -------
    sqi_values : np.ndarray
        SQI score per window (clipped to [0.1, 1.0]).
    sqi_times : np.ndarray
        Centre time of each window (seconds).
    """
    win_samples = int(window_s * fs)
    stride_samples = int(stride_s * fs)
    sqi_values, sqi_times = [], []
    for start in range(0, len(signal) - win_samples, stride_samples):
        window = signal[start : start + win_samples]
        k = kurtosis(window, fisher=True)
        sqi = max(0.1, min(1.0, 1.0 - (k - 2) / 10))
        sqi_values.append(sqi)
        sqi_times.append((start + win_samples / 2) / fs)
    return np.array(sqi_values), np.array(sqi_times)


def compute_windowed_f1(ref_peaks, test_peaks, fs,
                         window_s=10, stride_s=5):
    """Compute F1 score in sliding windows.

    Parameters
    ----------
    ref_peaks : np.ndarray
        Reference peak indices.
    test_peaks : np.ndarray
        Detected peak indices.
    fs : int
        Sampling frequency.
    window_s : float
        Window duration in seconds.
    stride_s : float
        Stride in seconds.

    Returns
    -------
    f1_values : np.ndarray
        Per-window F1 scores.
    f1_times : np.ndarray
        Centre time of each window (seconds).
    """
    win_samples = int(window_s * fs)
    stride_samples = int(stride_s * fs)
    f1_values, f1_times = [], []
    signal_length = max(
        np.max(ref_peaks) if len(ref_peaks) else 0,
        np.max(test_peaks) if len(test_peaks) else 0,
    ) + 1
    for start in range(0, signal_length - win_samples, stride_samples):
        end = start + win_samples
        ref_in = ref_peaks[(ref_peaks >= start) & (ref_peaks < end)]
        test_in = test_peaks[(test_peaks >= start) & (test_peaks < end)]
        if len(ref_in) == 0:
            f1_values.append(1.0 if len(test_in) == 0 else 0.0)
        else:
            result = evaluate_rpeaks(ref_in, test_in, fs)
            f1_values.append(result["F1"])
        f1_times.append((start + win_samples / 2) / fs)
    return np.array(f1_values), np.array(f1_times)

In [ ]:
# Compute SQI on BIOPAC (CH40 pipeline)
sqi_vals, sqi_times = compute_kurtosis_sqi(
    biopac_ecg_clean, biopac_fs, window_s=10, stride_s=5
)

# Windowed F1 — RPNet
rpnet_f1, rpnet_f1_times = compute_windowed_f1(
    ref_peaks, biopac_rpnet_peaks_original, biopac_fs,
    window_s=10, stride_s=5,
)

# Windowed F1 — Pan-Tompkins
pt_f1, pt_f1_times = compute_windowed_f1(
    ref_peaks, pt_peaks_biopac, biopac_fs,
    window_s=10, stride_s=5,
)

# Align lengths (use the shorter set)
n_common = min(len(sqi_vals), len(rpnet_f1), len(pt_f1))
sqi_aligned = sqi_vals[:n_common]
rpnet_f1_aligned = rpnet_f1[:n_common]
pt_f1_aligned = pt_f1[:n_common]

# --- Scatter plot ---
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(sqi_aligned, pt_f1_aligned, alpha=0.5, s=40, c="#1976D2",
           edgecolors="white", linewidth=0.5, label="Pan-Tompkins")
ax.scatter(sqi_aligned, rpnet_f1_aligned, alpha=0.5, s=40, c="#D32F2F",
           edgecolors="white", linewidth=0.5, label="RPNet")

ax.set_xlabel("Signal Quality Index (kurtosis-based)", fontsize=12)
ax.set_ylabel("Windowed F1 Score", fontsize=12)
ax.set_title("Signal Quality vs Detection F1 — RPNet vs Pan-Tompkins",
             fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 1.05)
ax.set_ylim(-0.05, 1.05)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "sqi_vs_f1_scatter.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved: {save_path}")
plt.show()

# Summary statistics
low_sqi_mask = sqi_aligned < 0.5
if np.any(low_sqi_mask):
    print(f"\nLow-SQI windows (SQI < 0.5): {np.sum(low_sqi_mask)}")
    print(f"  RPNet mean F1:         {rpnet_f1_aligned[low_sqi_mask].mean():.3f}")
    print(f"  Pan-Tompkins mean F1:  {pt_f1_aligned[low_sqi_mask].mean():.3f}")
else:
    print("\nNo low-SQI windows found (all windows have SQI >= 0.5).")

high_sqi_mask = sqi_aligned >= 0.5
if np.any(high_sqi_mask):
    print(f"\nHigh-SQI windows (SQI >= 0.5): {np.sum(high_sqi_mask)}")
    print(f"  RPNet mean F1:         {rpnet_f1_aligned[high_sqi_mask].mean():.3f}")
    print(f"  Pan-Tompkins mean F1:  {pt_f1_aligned[high_sqi_mask].mean():.3f}")

## 16. Final Workshop Summary

### Detector Hierarchy for Textile ECG

Based on findings across all six notebooks, the recommended detector
strategy depends on signal quality:

| Signal Quality | Recommended Detector | Rationale |
|---|---|---|
| **High SNR** | NeuroKit2 default (`neurokit`) | Fastest, accurate on clean signals |
| **Medium SNR** | PROMAC or consensus of multiple detectors | Ensemble voting reduces single-detector errors |
| **Low SNR** | RPNet (AI) or BIOPAC-weighted ensemble | Learned features handle morphology distortion |
| **Ground truth** | BIOPAC CH40 reference | Always available as gold standard |

### Recommendations for HRV Analysis from Textile ECG

1. **Always preprocess** — bandpass (0.5–40 Hz) + notch (60 Hz) + baseline
   removal before any detector.
2. **Use an ensemble or AI detector** for noisy segments — single classical
   detectors miss beats or hallucinate peaks in low-SNR windows.
3. **Cross-validate with BIOPAC** when available — the hardware-filtered
   CH40 channel provides a reliable ground truth.
4. **Check SQI before HRV** — exclude windows with kurtosis SQI < 0.3
   from HRV feature extraction (these segments likely have artifact-
   dominated morphology).
5. **Resample carefully** — use `resample_poly` (not simple decimation) to
   avoid aliasing when converting between sampling rates.
6. **Mind the refractory period** — enforce a minimum RR interval
   (≥300 ms / 200 BPM) as a post-processing step to catch double
   detections.

## 17. Performance Summary Table

Consolidated results table including RPNet alongside classical methods
from NB04.

In [ ]:
# Run several classical detectors for a comprehensive comparison
classical_methods = [
    "neurokit", "pantompkins1985", "hamilton2002",
    "elgendi2010", "rodrigues2021",
]

summary_rows = []

for method in tqdm(classical_methods, desc="Classical detectors"):
    try:
        t0 = time.perf_counter()
        _, info = nk.ecg_peaks(
            biopac_ecg_clean, sampling_rate=biopac_fs, method=method
        )
        elapsed = (time.perf_counter() - t0) * 1000
        peaks = info["ECG_R_Peaks"]
        metrics = evaluate_rpeaks(ref_peaks, peaks, biopac_fs)
        metrics["Method"] = method
        metrics["Time_ms"] = round(elapsed, 2)
        metrics["N_detected"] = len(peaks)
        summary_rows.append(metrics)
    except Exception as e:
        print(f"  {method} failed: {e}")

# Add RPNet
rpnet_row = biopac_rpnet_eval.copy()
rpnet_row["Method"] = "RPNet (IncUNet)"
rpnet_row["Time_ms"] = round(biopac_rpnet_time, 2)
rpnet_row["N_detected"] = len(biopac_rpnet_peaks_original)
summary_rows.append(rpnet_row)

summary_df = pd.DataFrame(summary_rows).set_index("Method")
col_order = ["N_detected", "TP", "FP", "FN", "Se", "PPV", "F1", "Time_ms"]
summary_df = summary_df[col_order].sort_values("F1", ascending=False)

print("\n" + "=" * 80)
print("BIOPAC CH40 (pipeline) — All Methods vs Reference (CH40)")
print("=" * 80)
print(f"Reference peaks: {len(ref_peaks)}")
print(summary_df.to_string())
print("=" * 80)

In [ ]:
# Heatmap of F1, Se, PPV across all methods
plot_df = summary_df[["Se", "PPV", "F1"]].astype(float)

fig, ax = plt.subplots(figsize=(8, max(4, len(plot_df) * 0.6)))
sns.heatmap(
    plot_df, annot=True, fmt=".3f", cmap="RdYlGn",
    vmin=0.8, vmax=1.0, linewidths=0.5, ax=ax,
)
ax.set_title("BIOPAC CH40 (pipeline) — Detection Metrics (all methods)", fontsize=13)
ax.set_ylabel("")

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "all_methods_heatmap.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved: {save_path}")
plt.show()

In [ ]:
# Timing bar chart — log scale
timing_df = summary_df[["Time_ms"]].sort_values("Time_ms")

fig, ax = plt.subplots(figsize=(10, max(4, len(timing_df) * 0.5)))
colors = ["#D32F2F" if "RPNet" in m else "#1976D2" for m in timing_df.index]
ax.barh(timing_df.index, timing_df["Time_ms"], color=colors, edgecolor="white")
ax.set_xlabel("Compute Time (ms)", fontsize=12)
ax.set_title("Compute Time — Classical vs Deep Learning", fontsize=13)
ax.set_xscale("log")

for i, (method, row) in enumerate(timing_df.iterrows()):
    ax.text(row["Time_ms"] * 1.1, i, f"{row['Time_ms']:.0f} ms",
            va="center", fontsize=10)

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, "timing_comparison.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
print(f"Saved: {save_path}")
plt.show()

## 18. Conclusion

This final notebook demonstrated how a deep learning approach — RPNet's
IncRes-UNet — can complement classical R-peak detectors for textile ECG
processing.

**Key takeaways:**

1. **Sampling rate alignment is non-negotiable** — RPNet expects 500 Hz
   input, and feeding signals at the wrong rate silently produces incorrect
   results.

2. **Distance transform framing** converts peak detection from a binary
   classification problem into a dense regression task, providing richer
   supervision signal during training and smoother output during inference.

3. **Windowed inference with overlap averaging** handles arbitrary-length
   recordings while mitigating boundary effects.

4. **No single detector is universally best** — the optimal choice depends
   on signal quality. For production textile ECG pipelines, consider an
   adaptive strategy that selects the detector based on per-window SQI.

5. **Computational cost matters** — RPNet is significantly slower than
   classical methods on CPU. For real-time applications, GPU acceleration
   or a lighter architecture may be necessary.

---

*End of ECG Signal Processing Workshop (Notebooks 1–6).*